In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


True Tesla T4


In [2]:
# Mount your Drive
from google.colab import drive
drive.mount('/content/drive')

# Your folder
BASE = '/content/drive/MyDrive/SentimentofHurricanes/Summer2026/Week3 Deliverables'

# Verify the path works — should list the folder contents
!ls "$BASE"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data  RoBERTa.ipynb


In [3]:
 !ls "$BASE/data"

facebook_comments_vader.csv	    reddit_relevant_posts_vader.csv
facebook_posts_vader.csv	    whitehouse_threads_comments_vader.csv
reddit_relevant_comments_vader.csv  whitehouse_threads_posts_vader.csv


In [6]:
!pip install -q transformers

In [4]:
import numpy as np, pandas as pd, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from scipy.special import softmax
from tqdm.auto import tqdm

MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"   # NOT the old non-latest checkpoint
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
config    = AutoConfig.from_pretrained(MODEL)                 # id2label: 0 negative, 1 neutral, 2 positive
model     = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)
model.eval()
print(config.id2label)
INPUT_FILES = [
      "facebook_posts_vader.csv",
      "facebook_comments_vader.csv",
      "reddit_relevant_posts_vader.csv",
      "reddit_relevant_comments_vader.csv",
      "whitehouse_threads_posts_vader.csv",
      "whitehouse_threads_comments_vader.csv",
  ]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

{0: 'negative', 1: 'neutral', 2: 'positive'}


In [5]:
def preprocess(text):
    out = []
    for t in str(text).split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http'  if t.startswith('http') else t
        out.append(t)
    return " ".join(out)

In [6]:
def score_df(df):
    texts = df['text'].fillna("").map(preprocess).tolist()
    BATCH = 128
    neg, neu, pos, truncated = [], [], [], []
    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i+BATCH]
        enc = tokenizer(batch, return_tensors='pt', padding=True,
                        truncation=True, max_length=512).to(device)
        full = tokenizer(batch, truncation=False)['input_ids']   # pre-truncation lengths
        truncated.extend([len(x) > 512 for x in full])
        with torch.no_grad():
            logits = model(**enc).logits.detach().cpu().numpy()
        probs = softmax(logits, axis=1)
        neg.extend(probs[:, 0]); neu.extend(probs[:, 1]); pos.extend(probs[:, 2])
    df = df.copy()
    df['roberta_neg'] = neg; df['roberta_neu'] = neu; df['roberta_pos'] = pos
    df['roberta_truncated'] = truncated
    df['roberta_label'] = np.array(['NEG', 'NEUTRAL', 'POS'])[
        np.argmax(df[['roberta_neg','roberta_neu','roberta_pos']].values, axis=1)]
    return df


In [11]:
IN_DIR  = f"{BASE}/data"
OUT_DIR = f"{BASE}/data"

for f in INPUT_FILES:
    in_csv  = f"{IN_DIR}/{f}"
    out_csv = f"{OUT_DIR}/{f.replace('.csv', '_roberta.csv')}"
    print(f"\n=== {f} ===")
    df = pd.read_csv(in_csv)
    df = score_df(df)
    df.to_csv(out_csv, index=False)
    print(f"rows: {len(df)} | truncated >512: {int(df['roberta_truncated'].sum())} | saved → {out_csv}")


=== facebook_posts_vader.csv ===


  0%|          | 0/8 [00:00<?, ?it/s]

rows: 952 | truncated >512: 37 | saved → /content/drive/MyDrive/SentimentofHurricanes/Summer2026/Week3 Deliverables/data/facebook_posts_vader_roberta.csv

=== facebook_comments_vader.csv ===


  0%|          | 0/467 [00:00<?, ?it/s]

rows: 59736 | truncated >512: 7 | saved → /content/drive/MyDrive/SentimentofHurricanes/Summer2026/Week3 Deliverables/data/facebook_comments_vader_roberta.csv

=== reddit_relevant_posts_vader.csv ===


...

rows: 120512 | truncated >512: 237 | saved → /content/drive/MyDrive/SentimentofHurricanes/Summer2026/Week3 Deliverables/data/reddit_relevant_comments_vader_roberta.csv

=== whitehouse_threads_posts_vader.csv ===


  0%|          | 0/1 [00:00<?, ?it/s]

rows: 13 | truncated >512: 2 | saved → /content/drive/MyDrive/SentimentofHurricanes/Summer2026/Week3 Deliverables/data/whitehouse_threads_posts_vader_roberta.csv

=== whitehouse_threads_comments_vader.csv ===


  0%|          | 0/18 [00:00<?, ?it/s]

rows: 2239 | truncated >512: 11 | saved → /content/drive/MyDrive/SentimentofHurricanes/Summer2026/Week3 Deliverables/data/whitehouse_threads_comments_vader_roberta.csv


In [12]:
import pandas as pd
FILES = ["facebook_posts","facebook_comments","reddit_relevant_posts",
           "reddit_relevant_comments","whitehouse_threads_posts","whitehouse_threads_comments"]
parts = []
for f in FILES:
    d = pd.read_csv(f"{BASE}/data/{f}_vader_roberta.csv")[['text','vader_label','roberta_label']]
    d['file'] = f
    parts.append(d)
qc = pd.concat(parts, ignore_index=True)
sample = qc.sample(20, random_state=1)
sample['agree'] = sample['vader_label'].str.lower().str[:3] == sample['roberta_label'].str.lower().str[:3]
print(sample.to_string())
import transformers, torch
print("\ntransformers", transformers.__version__, "| torch", torch.__version__,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           text vader_label roberta_label                      file  agree
89240   Bro, I’m an atheist too. Shut the fuck up and stop be